# YOGA Chatbot

## 0. Setup & Installation

In [1]:
# Install dependencies (Google Colab only)
!pip install -q gensim sastrawi tensorflow

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import json
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

# NLP
from gensim.models import Word2Vec
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Machine Learning
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout, 
    BatchNormalization, Activation
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("✓ All libraries imported successfully")

TensorFlow version: 2.20.0
✓ All libraries imported successfully


## 1. Pattern Acquisition

In [ ]:
# Load intents dataset (AUGMENTED VERSION)
with open('../data/intents_diy_augmented.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

intents = data['intents']

# Extract patterns and labels
patterns = []
labels = []
for intent in intents:
    for pattern in intent['patterns']:
        patterns.append(pattern)
        labels.append(intent['tag'])

df = pd.DataFrame({'pattern': patterns, 'label': labels})

print("=" * 70)
print("PATTERN ACQUISITION (AUGMENTED DATA)")
print("=" * 70)
print(f"Total intents: {len(intents)}")
print(f"Total patterns: {len(df)}")
print(f"Unique labels: {df['label'].nunique()}")
print(f"\nSample data:")
print(df.head(10))

## 2. Preprocessing

In [ ]:
# Initialize NLP tools
stemmer = StemmerFactory().create_stemmer()

def preprocess_text(text):
    """Preprocessing without stopword removal"""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    text = stemmer.stem(text)
    return text

# Apply preprocessing
df['processed_text'] = df['pattern'].apply(preprocess_text)
df['tokens'] = df['processed_text'].apply(lambda x: x.split() if x else [])

# Filter empty patterns
df = df[df['tokens'].str.len() > 0].reset_index(drop=True)

print("=" * 70)
print("PREPROCESSING (NO STOPWORD REMOVAL)")
print("=" * 70)
print(f"Patterns after filtering: {len(df)}")
print(f"\nSample preprocessing:")
for i in range(5):
    print(f"\n{i+1}. Original: {df['pattern'].iloc[i]}")
    print(f"   Processed: {df['processed_text'].iloc[i]}")
    print(f"   Tokens: {df['tokens'].iloc[i]}")

## 3. Feature Extraction (Word2Vec)

In [ ]:
# Train Word2Vec model
sentences = df['tokens'].tolist()

# Word2Vec hyperparameters
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 1
SG = 0  # CBOW
EPOCHS = 100

print("Training Word2Vec model...")
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=4,
    sg=SG,
    epochs=EPOCHS,
    seed=42
)

print(f"✓ Word2Vec trained: {len(w2v_model.wv)} words, {VECTOR_SIZE} dimensions")

# Determine max sequence length
sent_lengths = [len(sent) for sent in sentences]
MAX_LENGTH = int(np.percentile(sent_lengths, 95))

print(f"✓ Max sequence length: {MAX_LENGTH}")

In [ ]:
# Create sequence vectors for LSTM
def get_sequence_vector(tokens, w2v_model, max_len, vec_size):
    """Convert tokens to padded sequence of word vectors"""
    seq_vectors = []
    for word in tokens[:max_len]:
        if word in w2v_model.wv:
            seq_vectors.append(w2v_model.wv[word])
        else:
            seq_vectors.append(np.zeros(vec_size))
    
    # Pad with zeros
    while len(seq_vectors) < max_len:
        seq_vectors.append(np.zeros(vec_size))
    
    return np.array(seq_vectors)

# Create feature matrix X
print("Creating sequence vectors...")
X = np.array([
    get_sequence_vector(sent, w2v_model, MAX_LENGTH, VECTOR_SIZE) 
    for sent in sentences
])

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])

print("=" * 70)
print("FEATURE EXTRACTION")
print("=" * 70)
print(f"Feature matrix X: {X.shape}")
print(f"  - Samples: {X.shape[0]}")
print(f"  - Time steps: {X.shape[1]}")
print(f"  - Features: {X.shape[2]}")
print(f"\nLabel vector y: {y.shape}")
print(f"  - Classes: {len(label_encoder.classes_)}")

## 4. LSTM Modeling

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert labels to categorical
num_classes = len(label_encoder.classes_)
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print("=" * 70)
print("DATA SPLIT")
print("=" * 70)
print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Train ratio: {len(X_train)/(len(X_train)+len(X_test))*100:.1f}%")
print(f"Test ratio: {len(X_test)/(len(X_train)+len(X_test))*100:.1f}%")

In [ ]:
# Build LSTM model (UNIDIRECTIONAL)
def build_lstm_model(input_shape, num_classes):
    """Build Unidirectional LSTM model"""
    model = Sequential([
        # LSTM layer 1
        LSTM(128, return_sequences=True, input_shape=input_shape, name='lstm_1'),
        Dropout(0.3, name='dropout_1'),
        
        # LSTM layer 2
        LSTM(64, name='lstm_2'),
        Dropout(0.3, name='dropout_2'),
        
        # Dense layers
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.3, name='dropout_3'),
        
        # Output layer
        Dense(num_classes, activation='softmax', name='output')
    ], name='YOGA_LSTM')
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build model
input_shape = (X_train.shape[1], X_train.shape[2])
model = build_lstm_model(input_shape, num_classes)

print("=" * 70)
print("MODEL ARCHITECTURE (UNIDIRECTIONAL LSTM)")
print("=" * 70)
model.summary()

print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
# Define callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

In [ ]:
# Train model
print("=" * 70)
print("TRAINING MODEL")
print("=" * 70)

EPOCHS = 200
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training completed!")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax2.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

# Make predictions
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"\nTotal test samples: {len(y_test)}")
print(f"Correct predictions: {(y_pred == y_test).sum()}")
print(f"Wrong predictions: {(y_pred != y_test).sum()}")

In [ ]:
# Classification report
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (Top 10 Classes)")
print("=" * 70)

report = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True
)

# Convert to DataFrame and show top 10
report_df = pd.DataFrame(report).transpose()
report_df = report_df[report_df['support'] > 0].sort_values('f1-score', ascending=False)
print(report_df.head(10)[['precision', 'recall', 'f1-score', 'support']])

print(f"\n\nOverall Metrics:")
print(f"  Accuracy: {report['accuracy']*100:.2f}%")
print(f"  Macro Avg F1: {report['macro avg']['f1-score']*100:.2f}%")
print(f"  Weighted Avg F1: {report['weighted avg']['f1-score']*100:.2f}%")

In [ ]:
# Confusion matrix for most confused pairs
cm = confusion_matrix(y_test, y_pred)

# Find top 10 most confused pairs
confused_pairs = []
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i != j and cm[i, j] > 0:
            confused_pairs.append({
                'True': label_encoder.classes_[i],
                'Predicted': label_encoder.classes_[j],
                'Count': cm[i, j]
            })

confused_df = pd.DataFrame(confused_pairs).sort_values('Count', ascending=False)

print("\n" + "=" * 70)
print("TOP 10 MOST CONFUSED CLASS PAIRS")
print("=" * 70)
print(confused_df.head(10).to_string(index=False))

In [ ]:
# Confidence analysis
errors_mask = y_test != y_pred
error_confidences = y_pred_prob[errors_mask].max(axis=1)
correct_confidences = y_pred_prob[~errors_mask].max(axis=1)

print("\n" + "=" * 70)
print("CONFIDENCE ANALYSIS")
print("=" * 70)
print(f"\nCorrect predictions:")
print(f"  Avg confidence: {correct_confidences.mean():.4f}")
print(f"  Std: {correct_confidences.std():.4f}")
print(f"\nWrong predictions:")
print(f"  Avg confidence: {error_confidences.mean():.4f}")
print(f"  Std: {error_confidences.std():.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_confidences, bins=30, alpha=0.7, label='Correct', color='green', edgecolor='black')
ax.hist(error_confidences, bins=30, alpha=0.7, label='Wrong', color='red', edgecolor='black')
ax.set_xlabel('Prediction Confidence', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Inference Test

In [ ]:
def predict_intent(text):
    """Predict intent for input text"""
    processed = preprocess_text(text)
    tokens = processed.split() if processed else []
    
    seq_vec = get_sequence_vector(tokens, w2v_model, MAX_LENGTH, VECTOR_SIZE)
    seq_vec = seq_vec.reshape(1, MAX_LENGTH, VECTOR_SIZE)
    
    pred_prob = model.predict(seq_vec, verbose=0)[0]
    pred_class = np.argmax(pred_prob)
    confidence = pred_prob[pred_class]
    intent = label_encoder.classes_[pred_class]
    
    top_3_idx = np.argsort(pred_prob)[-3:][::-1]
    top_3 = [(label_encoder.classes_[i], pred_prob[i]) for i in top_3_idx]
    
    return intent, confidence, top_3

# Test samples
test_inputs = [
    "halo",
    "selamat pagi",
    "terima kasih",
    "wisata di yogyakarta",
    "tempat wisata sleman",
    "wisata kecamatan prambanan",
    "rekomendasi kabupaten bantul"
]

print("=" * 70)
print("INFERENCE TEST")
print("=" * 70)

for text in test_inputs:
    intent, conf, top_3 = predict_intent(text)
    print(f"\nInput: '{text}'")
    print(f"Predicted: {intent} (confidence: {conf:.4f})")
    print(f"Top 3:")
    for i, (label, prob) in enumerate(top_3, 1):
        print(f"  {i}. {label}: {prob:.4f}")

## 7. Summary

In [ ]:
print("=" * 70)
print("YOGA CHATBOT - FINAL SUMMARY")
print("=" * 70)

print("\nDataset:")
print(f"  Total patterns: {len(df)}")
print(f"  Total intents: {len(label_encoder.classes_)}")
print(f"  Train samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")

print("\nPreprocessing:")
print(f"  Lowercase + Stemming (NO Stopword Removal)")
print(f"  Vocabulary size: {len(w2v_model.wv)} words")

print("\nFeature Extraction:")
print(f"  Method: Word2Vec (CBOW)")
print(f"  Vector size: {VECTOR_SIZE}")
print(f"  Max sequence length: {MAX_LENGTH}")
print(f"  Feature shape: {X.shape}")

print("\nModel Architecture:")
print(f"  Type: Bidirectional LSTM")
print(f"  LSTM layers: 2 (128, 64 units)")
print(f"  Dropout: 0.3")
print(f"  Total parameters: {model.count_params():,}")

print("\nTraining:")
print(f"  Epochs: {len(history.history['accuracy'])}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Optimizer: Adam")
print(f"  Final train accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"  Final val accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")

print("\nFinal Test Performance:")
print(f"  Test Accuracy: {test_accuracy*100:.2f}%")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Macro F1-Score: {report['macro avg']['f1-score']*100:.2f}%")
print(f"  Weighted F1-Score: {report['weighted avg']['f1-score']*100:.2f}%")

print("\nModel Status: READY FOR DEPLOYMENT")
print("=" * 70)